In [1]:
import sys
print(sys.executable)

C:\Users\chami\sepsis-early-warning\.venv\Scripts\python.exe


In [2]:
from pathlib import Path
import pandas as pd

DATA = Path("../data/raw")   

files_a = sorted((DATA / "training_setA").glob("*.psv"))
files_b = sorted((DATA / "training_setB").glob("*.psv"))

print("Hospital A files:", len(files_a))
print("Hospital B files:", len(files_b))
print([f.name for f in files_a[:5]])

Hospital A files: 20336
Hospital B files: 20000
['p000001.psv', 'p000002.psv', 'p000003.psv', 'p000004.psv', 'p000005.psv']


In [3]:
df = pd.read_csv(files_a[0], sep="|")   # PhysioNet files use | as the separator

print("Shape (rows, columns):", df.shape)
df.head(10)

Shape (rows, columns): (54, 41)


,HR,O2Sat,Temp,SBP,MAP,DBP,Resp,EtCO2,BaseExcess,HCO3,...,WBC,Fibrinogen,Platelets,Age,Gender,Unit1,Unit2,HospAdmTime,ICULOS,SepsisLabel
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,83.14,0,NaN,NaN,-0.03,1,0
1,97.0,95.0,NaN,98.0,75.33,NaN,19.0,NaN,NaN,NaN,...,NaN,NaN,NaN,83.14,0,NaN,NaN,-0.03,2,0
2,89.0,99.0,NaN,122.0,86.00,NaN,22.0,NaN,NaN,NaN,...,NaN,NaN,NaN,83.14,0,NaN,NaN,-0.03,3,0
3,90.0,95.0,NaN,NaN,NaN,NaN,30.0,NaN,24.0,NaN,...,NaN,NaN,NaN,83.14,0,NaN,NaN,-0.03,4,0
4,103.0,88.5,NaN,122.0,91.33,NaN,24.5,NaN,NaN,NaN,...,NaN,NaN,NaN,83.14,0,NaN,NaN,-0.03,5,0
5,110.0,91.0,NaN,NaN,NaN,NaN,22.0,NaN,NaN,NaN,...,NaN,NaN,NaN,83.14,0,NaN,NaN,-0.03,6,0
6,108.0,92.0,36.11,123.0,77.00,NaN,29.0,NaN,NaN,NaN,...,NaN,NaN,NaN,83.14,0,NaN,NaN,-0.03,7,0
7,106.0,90.5,NaN,93.0,76.33,NaN,29.0,NaN,NaN,NaN,...,NaN,NaN,NaN,83.14,0,NaN,NaN,-0.03,8,0
8,104.0,95.0,NaN,133.0,88.33,NaN,26.0,NaN,NaN,NaN,...,NaN,NaN,NaN,83.14,0,NaN,NaN,-0.03,9,0
9,102.0,91.0,NaN,134.0,87.33,NaN,30.0,NaN,NaN,NaN,...,NaN,NaN,NaN,83.14,0,NaN,NaN,-0.03,10,0


In [4]:
VITALS = ["HR", "O2Sat", "Temp", "SBP", "MAP", "DBP", "Resp", "EtCO2"]
LABS = ["BaseExcess", "HCO3", "FiO2", "pH", "PaCO2", "SaO2", "AST", "BUN",
        "Alkalinephos", "Calcium", "Chloride", "Creatinine", "Bilirubin_direct",
        "Glucose", "Lactate", "Magnesium", "Phosphate", "Potassium",
        "Bilirubin_total", "TroponinI", "Hct", "Hgb", "PTT", "WBC",
        "Fibrinogen", "Platelets"]
DEMO = ["Age", "Gender", "Unit1", "Unit2", "HospAdmTime", "ICULOS"]
LABEL = "SepsisLabel"

print(len(VITALS), "vitals,", len(LABS), "labs,", len(DEMO), "demographic/admin")

# Sanity check
grouped = set(VITALS + LABS + DEMO + [LABEL])
print("In file but not grouped:", set(df.columns) - grouped)
print("Grouped but not in file:", grouped - set(df.columns))

8 vitals, 26 labs, 6 demographic/admin
In file but not grouped: set()
Grouped but not in file: set()


In [5]:
print("First ICULOS:", df["ICULOS"].iloc[0])
print("Step sizes between rows:")
print(df["ICULOS"].diff().value_counts(dropna=False))

First ICULOS: 1
Step sizes between rows:
ICULOS
1.0    53
NaN     1
Name: count, dtype: int64


In [6]:
STATIC = ["Age", "Gender", "Unit1", "Unit2", "HospAdmTime"]
print(df[STATIC].nunique(dropna=False))  

Age            1
Gender         1
Unit1          1
Unit2          1
HospAdmTime    1
dtype: int64


In [7]:
#One row summary of a patient file for EDA
def summarize(path):
    p = pd.read_csv(path, sep="|")
    septic = p[LABEL].max() == 1
    first_one = p.loc[p[LABEL] == 1, "ICULOS"].min() if septic else None
    return {
        "patient_id": path.stem,              
        "n_hours": len(p),
        "septic": septic,
        "first_label_iculos": first_one,
        "vital_missing": p[VITALS].isna().mean().mean(),
        "lab_missing": p[LABS].isna().mean().mean(),
    }

summary = pd.DataFrame([summarize(f) for f in files_a[:500]])

print("Septic fraction in first 500:", round(summary["septic"].mean(), 3))
summary.head()

Septic fraction in first 500: 0.09


,patient_id,n_hours,septic,first_label_iculos,vital_missing,lab_missing
0,p000001,54,False,NaN,0.451389,0.959402
1,p000002,23,False,NaN,0.342391,0.978261
2,p000003,48,False,NaN,0.265625,0.954327
3,p000004,29,False,NaN,0.344828,0.952255
4,p000005,48,False,NaN,0.697917,0.959936


In [8]:
septic_ids = summary.loc[summary["septic"], "patient_id"].head(3).tolist()

for pid in septic_ids:
    p = pd.read_csv(DATA / "training_setA" / f"{pid}.psv", sep="|")
    first = p.loc[p[LABEL] == 1, "ICULOS"].min()
    after = p[p["ICULOS"] >= first]
    print(f"{pid}: stay = {len(p)} h, first label=1 at ICULOS {first}, "
          f"stays 1 afterwards: {(after[LABEL] == 1).all()}, "
          f"hours from first 1 to end: {len(after)}")

# Count how many septic patients are labeled 1 from the very first hour
print("\nLabeled 1 at hour 1:", (summary["first_label_iculos"] == 1).sum())

p000009: stay = 258 h, first label=1 at ICULOS 249, stays 1 afterwards: True, hours from first 1 to end: 10
p000011: stay = 34 h, first label=1 at ICULOS 26, stays 1 afterwards: True, hours from first 1 to end: 10
p000015: stay = 15 h, first label=1 at ICULOS 6, stays 1 afterwards: True, hours from first 1 to end: 10

Labeled 1 at hour 1: 4


In [9]:
p = pd.read_csv(DATA / "training_setA" / "p000011.psv", sep="|")
print("First ICULOS:", p["ICULOS"].iloc[0], " Last ICULOS:", p["ICULOS"].iloc[-1], " Rows:", len(p))
print(p["ICULOS"].diff().value_counts(dropna=False))

First ICULOS: 2  Last ICULOS: 35  Rows: 34
ICULOS
1.0    33
NaN     1
Name: count, dtype: int64


In [10]:
summary.groupby("septic")[["n_hours", "vital_missing", "lab_missing"]].mean().round(2)

,n_hours,vital_missing,lab_missing
septic,,,
False,36.78,0.34,0.93
True,59.73,0.33,0.90


In [11]:
summary.groupby("septic")["n_hours"].describe()

,count,mean,std,min,25%,50%,75%,max
septic,,,,,,,,
False,455.0,36.782418,12.687489,9.0,25.0,39.0,47.0,59.0
True,45.0,59.733333,54.865290,8.0,18.0,44.0,84.0,258.0


## Findings (Part C: exploration of individual patients and a 500-patient sample)

**Structure**
- Hospital A: ___ patients, Hospital B: ___ patients (check: Cell 2; expected 20,336 and 20,000)
- One file per patient; one row per hour of ICU stay
- 41 columns: 8 vitals, 26 labs, 6 demographic/admin, 1 label (SepsisLabel)
- Constant during a stay: ___ (check: Cell 7; expected Age, Gender, Unit1, Unit2, HospAdmTime)

**Time**
- Not every record starts at ICULOS 1 (e.g. p000011 starts at hour 2 and runs to 35 with no gaps)
- Rows ≠ hours: use ICULOS values, not row positions, for time-based features
- To verify in SQL: how many records start after hour 1, and whether any have gaps

**Labels**
- SepsisLabel turns on 6 hours before sepsis onset and stays 1 until the end of the record
- In all 3 septic examples, the record ends exactly 10 rows after the first label of 1 (about 3 hours after onset)
- Hypothesis: septic records are truncated shortly after onset. Total record length is therefore leakage and must never be a feature
- 4 of 500 patients are labeled septic from their first hour, so there's nothing to predict in advance for them (cohort decision in Part F)

**Class balance (500-patient sample, Hospital A)**
- 45 of 500 patients septic (9%)

**Length of stay**
- Non-septic: median 39 h, max 59 h. Septic: median 44 h, max 258 h
- Hypothesis: non-septic records are capped at around 60 h. If true, any hour beyond ~60 belongs only to septic patients, so ICULOS becomes a shortcut feature
- Plan: verify in SQL; compare models with and without ICULOS; watch for ICULOS in SHAP

**Missingness (patient-averaged, 500-patient sample)**
- Vitals missing ~33–34% of hours; labs ~90–93%
- Septic patients have slightly more labs measured (90% vs 93% missing)
- Missingness is informative (labs are ordered when clinicians are concerned), so add missingness flags and "hours since last measured" features
- To check: does lab ordering increase in the hours before onset?